In [5]:
import polars as pl
from tcrtrifold.tcrdock_utils import (
    dgeom_ndarr_from_dgeom_series,
    mn_distr_from_dgeom_ndarr,
    mn_distance_from,
    un_cossin_embed,
)
from tcrtrifold.utils import filter_to_cog_thresh, FORMAT_ANTIGEN_COLS, FORMAT_TCR_COLS

assay_type = pl.read_parquet("../../data/iedb_meta/assay_type.parquet")

iedb_II_conf = (
    pl.read_parquet("../../data/iedb_II/triad/iedb_II_triad.conf_af3.parquet")
    .drop(["receptor_id", "references"])
    .join(
        pl.read_parquet(
            "../../data/iedb_II/triad/iedb_II_triad.receptor_reference.parquet"
        ).select("job_name", "receptor_id", "references"),
        on="job_name",
        how="left",
    )
)
iedb_II = (
    (
        iedb_II_conf.explode("references", "receptor_id")
        .filter(
            (pl.col("references").is_null()) | (pl.col("references").list.len() == 1)
        )
        .explode("references")
        .explode("receptor_id")
    )
    .join(
        assay_type,
        on=["references", "receptor_id", "peptide", "mhc_1_name", "mhc_2_name"],
        how="left",
    )
    .filter(~pl.col("cognate") | pl.col("assay_type").is_not_null())
)


iedb_I_conf = (
    pl.read_parquet("../../data/iedb_I/triad/iedb_I_triad.conf_af3.parquet")
    .drop(["receptor_id", "references"])
    .join(
        pl.read_parquet(
            "../../data/iedb_I/triad/iedb_I_triad.receptor_reference.parquet"
        ).select("job_name", "receptor_id", "references"),
        on="job_name",
        how="left",
    )
)
iedb_I = (
    (
        iedb_I_conf.explode("references", "receptor_id")
        .filter(
            (pl.col("references").is_null()) | (pl.col("references").list.len() == 1)
        )
        .explode("references")
        .explode("receptor_id")
    )
    .join(
        assay_type,
        on=["references", "receptor_id", "peptide", "mhc_1_name", "mhc_2_name"],
        how="left",
    )
    .filter(~pl.col("cognate") | pl.col("assay_type").is_not_null())
)

# REMOVEME TO ADD BACK IN
iedb_II_annot = pl.read_parquet(
    "../../data/iedb_II_full/triad/iedb_II_triad.annotated.parquet"
).filter(pl.col("triad_in_pdb"))

iedb_II = iedb_II.join(iedb_II_annot, on="job_name", how="anti")

# REMOVEME TO ADD BACK IN
iedb_I_annot = pl.read_parquet(
    "../../data/iedb_I_full/triad/iedb_I_triad.annotated.parquet"
).filter(pl.col("triad_in_pdb"))


iedb_I = iedb_I.join(iedb_I_annot, on="job_name", how="anti")

# additionally filter out
iedb_I = iedb_I.filter(pl.col("job_name") != "121f50b609621ebe92777820dad90eb9")

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score


def study_differences(df, featname):

    dfs = []

    antigen_assay_p = (
        df.filter(pl.col("cognate"))
        .group_by(FORMAT_ANTIGEN_COLS + ["assay_type", "references"])
        .agg(pl.col("job_name").unique())
        .filter(pl.col("job_name").list.len() >= 3)
        .partition_by(FORMAT_ANTIGEN_COLS + ["assay_type"])
    )

    antigen_assay_p = [p for p in antigen_assay_p if p.height > 1]

    for p in antigen_assay_p:
        antigen = p.select(FORMAT_ANTIGEN_COLS)[0]
        negs = (
            df.filter(~pl.col("cognate"))
            .join(antigen, on=FORMAT_ANTIGEN_COLS)
            .select(featname)
            .to_series()
            .to_numpy()
        )

        aucs = []
        study = []
        tcr_cnts = []
        feats = []
        for row in p.partition_by("references"):

            pos = (
                df.join(
                    row.explode("job_name").select("job_name").unique(), on="job_name"
                )
                .select("job_name", featname)
                .unique()
                .select(featname)
                .to_series()
                .to_numpy()
            )

            dat = np.concat([pos, negs])
            y = np.concat([np.full(pos.shape, True), np.full(negs.shape, False)])

            aucs.append(roc_auc_score(y, dat))
            study.append(row["references"].item())
            tcr_cnts.append(row.explode("job_name").select("job_name").unique().height)
            feats.append(pos.tolist())

        dfs.append(
            {
                "auc": aucs,
                "study": study,
                "tcr_counts": tcr_cnts,
                "peptide": antigen["peptide"].item(),
                "mhc_1_name": antigen["mhc_1_name"].item(),
                "mhc_2_name": antigen["mhc_2_name"].item(),
                "feats": feats,
            }
        )

    comp = pl.DataFrame(dfs)
    return comp

: 

In [ ]:
from scipy.stats import kruskal

featname = "mean_p_tcr_interface_pae"

iedb_I_sd = study_differences(iedb_I, featname).with_columns(
    pl.col("feats")
    .map_elements(lambda x: kruskal(*x).pvalue, return_dtype=pl.Float64)
    .alias("kruskal_pvalue")
)
iedb_II_sd = study_differences(iedb_II, featname).with_columns(
    pl.col("feats")
    .map_elements(lambda x: kruskal(*x).pvalue, return_dtype=pl.Float64)
    .alias("kruskal_pvalue")
)

In [9]:

iedb_II_sd.with_columns(
    pl.col("feats")
    .map_elements(lambda x: kruskal(*x).pvalue, return_dtype=pl.Float64)
    .alias("kruskal_pvalue")
)

auc,study,tcr_counts,peptide,mhc_1_name,mhc_2_name,feats,kruskal_pvalue
list[f64],list[str],list[i64],str,str,str,list[list[f64]],f64
"[0.423627, 0.401117]","[""1040829"", ""1040295""]","[226, 378]","""TFEYVSQPFLMDLE""","""DPA1*01:03""","""DPB1*04:01""","[[13.499483, 11.015476, … 21.596872], [19.537735, 9.04648, … 5.986916]]",0.23664
